# Manual keyboard balancing and data logger

Use the keyboard to manually move the rotary arm while trying to keep the pendulum upright.

**Controls**
- `Left` / `A`: negative motor velocity
- `Right` / `D`: positive motor velocity
- `Up` / `W`: increase command magnitude
- `Down` / `S`: decrease command magnitude
- `Space`: immediate hard stop and zero command
- `Esc`: stop the experiment, save data, and generate plots

The notebook logs every observation returned by the firmware and every control input sent to it. Data are saved as CSV and plots as PNG when the run ends.

> Safety: start with a small command speed and keep hands clear of the mechanism.


In [ ]:
from control_comms import ControlComms, StatusCode, DebugLevel
from pynput import keyboard
from pathlib import Path
from datetime import datetime
import threading
import time
import math
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Connection and experiment settings
SERIAL_PORT = "COM6"      # Change for your machine
BAUD_RATE = 500000
TIMEOUT = 0.25

# Firmware command IDs
CMD_SET_HOME = 0
CMD_MOVE_TO = 1
CMD_MOVE_BY = 2
CMD_SET_STEP_MODE = 3
CMD_SET_VELOCITY = 4
CMD_HARD_STOP = 5
CMD_QUERY = 6
CMD_RESET_SAFETY = 7

STEP_MODE_16 = 4

# Manual-control settings
INITIAL_SPEED_PPS = 250.0
SPEED_STEP_PPS = 50.0
MAX_MANUAL_SPEED_PPS = 1200.0
CONTROL_PERIOD_S = 0.02     # target logging/query period (~50 Hz)
CONTROL_SIGN = 1.0          # change to -1 if left/right direction is reversed

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


In [ ]:
# Connect
ctrl = ControlComms(timeout=TIMEOUT, debug_level=DebugLevel.DEBUG_ERROR)
ret = ctrl.connect(SERIAL_PORT, BAUD_RATE)
if ret is not StatusCode.OK:
    raise RuntimeError(f"Could not connect to {SERIAL_PORT}")

# Put the firmware in a known state
ctrl.step(CMD_HARD_STOP, [0.0])
ctrl.step(CMD_SET_STEP_MODE, [STEP_MODE_16])
ctrl.step(CMD_SET_HOME, [0.0])
ctrl.step(CMD_RESET_SAFETY, [0.0])

print("Connected.")
print("Use Left/Right or A/D to move, Up/Down or W/S to change speed.")
print("Space = hard stop, Esc = finish/save.")


In [ ]:
# Shared keyboard state
state = {
    "left": False,
    "right": False,
    "speed_pps": INITIAL_SPEED_PPS,
    "stop_requested": False,
    "emergency_stop": False,
}
state_lock = threading.Lock()

def _set_motion_key(name, pressed):
    with state_lock:
        state[name] = pressed

def on_press(key):
    try:
        ch = key.char.lower()
    except AttributeError:
        ch = None

    if key == keyboard.Key.left or ch == "a":
        _set_motion_key("left", True)
    elif key == keyboard.Key.right or ch == "d":
        _set_motion_key("right", True)
    elif key == keyboard.Key.up or ch == "w":
        with state_lock:
            state["speed_pps"] = min(MAX_MANUAL_SPEED_PPS, state["speed_pps"] + SPEED_STEP_PPS)
            print(f"speed = {state['speed_pps']:.0f} pps")
    elif key == keyboard.Key.down or ch == "s":
        with state_lock:
            state["speed_pps"] = max(0.0, state["speed_pps"] - SPEED_STEP_PPS)
            print(f"speed = {state['speed_pps']:.0f} pps")
    elif key == keyboard.Key.space:
        with state_lock:
            state["emergency_stop"] = True
            state["left"] = False
            state["right"] = False
        try:
            ctrl.step(CMD_HARD_STOP, [0.0])
        except Exception:
            pass
        print("HARD STOP")
    elif key == keyboard.Key.esc:
        with state_lock:
            state["stop_requested"] = True
        return False

def on_release(key):
    try:
        ch = key.char.lower()
    except AttributeError:
        ch = None
    if key == keyboard.Key.left or ch == "a":
        _set_motion_key("left", False)
    elif key == keyboard.Key.right or ch == "d":
        _set_motion_key("right", False)


In [ ]:
def wrapped_error_deg(angle_deg, setpoint_deg=180.0):
    return (setpoint_deg - angle_deg + 180.0) % 360.0 - 180.0

records = []
session_start_wall = time.time()
session_start_perf = time.perf_counter()
prev_host_t = None
prev_pendulum = None
prev_rotor = None
last_sent_velocity = None

listener = keyboard.Listener(on_press=on_press, on_release=on_release)
listener.start()

print("Manual balancing started. Press Esc to finish.")

try:
    while True:
        loop_start = time.perf_counter()

        with state_lock:
            left = state["left"]
            right = state["right"]
            speed_pps = state["speed_pps"]
            stop_requested = state["stop_requested"]
            estop = state["emergency_stop"]

        if stop_requested:
            break

        if estop:
            requested_velocity = 0.0
        elif left and not right:
            requested_velocity = -CONTROL_SIGN * speed_pps
        elif right and not left:
            requested_velocity = CONTROL_SIGN * speed_pps
        else:
            requested_velocity = 0.0

        # Send a new velocity only when the requested input changes.
        # Otherwise query the board so observations continue to be sampled.
        if last_sent_velocity is None or requested_velocity != last_sent_velocity:
            command_sent = CMD_SET_VELOCITY if requested_velocity != 0.0 else CMD_HARD_STOP
            action_sent = requested_velocity if requested_velocity != 0.0 else 0.0
            resp = ctrl.step(command_sent, [action_sent])
            last_sent_velocity = requested_velocity
        else:
            command_sent = CMD_QUERY
            action_sent = requested_velocity
            resp = ctrl.step(CMD_QUERY, [0.0])

        host_elapsed = time.perf_counter() - session_start_perf
        host_unix = time.time()

        if resp is not None:
            status, mcu_timestamp_ms, terminated, obs = resp
            pendulum_deg = float(obs[0]) if len(obs) > 0 else math.nan
            rotor_deg = float(obs[1]) if len(obs) > 1 else math.nan
            motor_speed_pps = float(obs[2]) if len(obs) > 2 else math.nan
            l6474_status = int(obs[3]) if len(obs) > 3 else -1

            dt = math.nan if prev_host_t is None else host_elapsed - prev_host_t
            pendulum_velocity_dps = math.nan
            rotor_velocity_dps = math.nan
            if prev_host_t is not None and dt > 0:
                dp = (pendulum_deg - prev_pendulum + 180.0) % 360.0 - 180.0
                pendulum_velocity_dps = dp / dt
                rotor_velocity_dps = (rotor_deg - prev_rotor) / dt

            records.append({
                "host_unix_s": host_unix,
                "host_elapsed_s": host_elapsed,
                "mcu_timestamp_ms": int(mcu_timestamp_ms),
                "mcu_elapsed_s": (int(mcu_timestamp_ms) - records[0]["mcu_timestamp_ms"]) / 1000.0 if records else 0.0,
                "response_status": int(status),
                "terminated": bool(terminated),
                "pendulum_angle_deg": pendulum_deg,
                "pendulum_error_deg": wrapped_error_deg(pendulum_deg),
                "pendulum_velocity_dps_est": pendulum_velocity_dps,
                "rotor_angle_deg": rotor_deg,
                "rotor_velocity_dps_est": rotor_velocity_dps,
                "motor_speed_pps_observed": motor_speed_pps,
                "l6474_status_raw": l6474_status,
                "command_id_sent": int(command_sent),
                "command_velocity_pps": float(requested_velocity),
                "manual_speed_setting_pps": float(speed_pps),
                "key_left": bool(left),
                "key_right": bool(right),
                "emergency_stop": bool(estop),
            })

            prev_host_t = host_elapsed
            prev_pendulum = pendulum_deg
            prev_rotor = rotor_deg

            if terminated:
                print("Firmware reported termination/safety condition.")
                break

        elapsed = time.perf_counter() - loop_start
        if elapsed < CONTROL_PERIOD_S:
            time.sleep(CONTROL_PERIOD_S - elapsed)

finally:
    try:
        ctrl.step(CMD_HARD_STOP, [0.0])
    except Exception:
        pass
    try:
        listener.stop()
    except Exception:
        pass

print(f"Stopped. Samples collected: {len(records)}")


In [ ]:
# Save all measured observations and control inputs
df = pd.DataFrame(records)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = DATA_DIR / f"manual_balance_{stamp}.csv"
png_path = DATA_DIR / f"manual_balance_{stamp}.png"

df.to_csv(csv_path, index=False)
print(f"CSV saved: {csv_path.resolve()}")

display(df.head())
display(df.tail())
print(df.describe(include="all"))


In [ ]:
# Plot the full experiment
if len(df) == 0:
    raise RuntimeError("No samples were recorded.")

t = df["host_elapsed_s"]

fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

axes[0].plot(t, df["pendulum_angle_deg"], label="pendulum angle")
axes[0].axhline(180.0, linestyle="--", label="upright setpoint")
axes[0].set_ylabel("Pendulum (deg)")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(t, df["pendulum_error_deg"], label="pendulum error")
axes[1].plot(t, df["pendulum_velocity_dps_est"], label="pendulum velocity est.")
axes[1].set_ylabel("deg / deg/s")
axes[1].legend()
axes[1].grid(True)

axes[2].plot(t, df["rotor_angle_deg"], label="rotor angle")
axes[2].plot(t, df["rotor_velocity_dps_est"], label="rotor velocity est.")
axes[2].set_ylabel("deg / deg/s")
axes[2].legend()
axes[2].grid(True)

axes[3].plot(t, df["command_velocity_pps"], label="command velocity")
axes[3].plot(t, df["motor_speed_pps_observed"], label="observed motor speed")
axes[3].set_ylabel("pps")
axes[3].legend()
axes[3].grid(True)

axes[4].plot(t, df["l6474_status_raw"], label="L6474 status raw")
axes[4].plot(t, df["response_status"], label="firmware response status")
axes[4].set_ylabel("status")
axes[4].set_xlabel("Host elapsed time (s)")
axes[4].legend()
axes[4].grid(True)

fig.suptitle("Manual inverted-pendulum balancing experiment")
fig.tight_layout()
fig.savefig(png_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"Plot saved: {png_path.resolve()}")


In [ ]:
# Optional: inspect key/control transitions only
control_changes = df.loc[df["command_velocity_pps"].ne(df["command_velocity_pps"].shift())]
display(control_changes[[
    "host_elapsed_s",
    "command_velocity_pps",
    "manual_speed_setting_pps",
    "key_left",
    "key_right",
    "pendulum_angle_deg",
    "rotor_angle_deg",
    "motor_speed_pps_observed",
    "l6474_status_raw",
]])
